In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import sys

GOOGLE_DRIVE_PATH_AFTER_MYDRIVE = 'RL'
GOOGLE_DRIVE_PATH = os.path.join('drive', 'My Drive', GOOGLE_DRIVE_PATH_AFTER_MYDRIVE)
print(os.listdir(GOOGLE_DRIVE_PATH))

# Add Google Drive path to sys.path
sys.path.insert(0, GOOGLE_DRIVE_PATH)

['test.py', 'evaluate.py', 'train.py', 'agents', 'games']


In [ ]:
import numpy as np
import torch
import random
import time

from games import TicTacToe
from agents import MuZeroAgent, AlphaZeroAgent, MinimaxAgent

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def evaluate_vs_random(agent, game_class, num_games=20, agent_first=None):
    """Evaluate agent against random opponent."""
    results = {'win': 0, 'loss': 0, 'draw': 0}
    for _ in range(num_games):
        game = game_class()
        if agent_first is None:
            agent_player = random.choice([1, 2])
        else:
            agent_player = 1 if agent_first else 2
        while game.result() is None:
            if game.current_player_id == agent_player:
                probs = agent.select_action(game, temperature=0)
                action = np.argmax(probs)
            else:
                action = random.choice(game.legal_actions())
            game.step(action)
        r = game.result()
        if (r == 1 and agent_player == 1) or (r == -1 and agent_player == 2):
            results['win'] += 1
        elif r == 0:
            results['draw'] += 1
        else:
            results['loss'] += 1
    return results

def play_match(agent1, agent2, game_class, num_games=20, temperature=0.0, verbose=False):
    """Play matches. Agents swap sides each game."""
    results = {'agent1_wins': 0, 'agent2_wins': 0, 'draws': 0,
               'agent1_time': 0, 'agent2_time': 0, 'total_moves': 0}

    for game_num in range(num_games):
        game = game_class()
        # Even games: agent1=P1, agent2=P2. Odd games: swapped.
        agents = [agent1, agent2] if game_num % 2 == 0 else [agent2, agent1]
        names = ['Agent1', 'Agent2'] if game_num % 2 == 0 else ['Agent2', 'Agent1']

        if verbose:
            print(f"\nGame {game_num + 1}: {names[0]} (P1) vs {names[1]} (P2)")

        move_count = 0
        while game.result() is None:
            current = game.current_player() - 1
            agent = agents[current]
            name = names[current]

            start_time = time.time()
            if hasattr(agent, 'select_action'):
                probs = agent.select_action(game, temperature=temperature)
                if isinstance(probs, tuple):
                    probs = probs[0]
                action = np.argmax(probs) if temperature == 0 else np.random.choice(len(probs), p=probs)
            else:
                action = agent.search(game)
            elapsed = time.time() - start_time

            # Track time by agent identity, not player number
            if name == 'Agent1':
                results['agent1_time'] += elapsed
            else:
                results['agent2_time'] += elapsed

            game.step(action)
            move_count += 1
            if verbose:
                print(f"Move {move_count}: {name} plays {action}")
                print(game.render())

        results['total_moves'] += move_count
        outcome = game.result()
        if outcome == 0:
            results['draws'] += 1
        elif outcome == 1:
            # P1 won
            if game_num % 2 == 0:
                results['agent1_wins'] += 1
            else:
                results['agent2_wins'] += 1
        else:
            # P2 won
            if game_num % 2 == 0:
                results['agent2_wins'] += 1
            else:
                results['agent1_wins'] += 1

    return results

set_seed(42)


In [ ]:
MMagent = MinimaxAgent(TicTacToe, max_depth=9, use_tt=True, tt_size=1e6)
print(f"1. Constructor: action_size={MMagent.action_size}, max_depth={MMagent.max_depth}")

# select_action works identically
game = TicTacToe()
probs, root = MMagent.select_action(game, return_root=True)
print(f"2. select_action: probs shape={probs.shape}, best={probs.argmax()}, root={root}")

print("--- MMagent vs Random ---")
print(evaluate_vs_random(MMagent, TicTacToe, 20))

1. Constructor: action_size=9, max_depth=9
2. select_action: probs shape=(9,), best=4, root=None
--- MMagent vs Random ---
{'win': 17, 'loss': 0, 'draw': 3}


In [ ]:
az_agent = AlphaZeroAgent(
    TicTacToe,
    num_simulations=50,
    lr=0.001,
    batch_size=32,
    num_blocks=2,
    channels=32
)

print("--- AlphaZero: Before Training (vs Random) ---")
print(evaluate_vs_random(az_agent, TicTacToe, 20))

print("--- AlphaZero: Training ---")
az_agent.train(num_epochs=30, games_per_epoch=30, steps_per_epoch=100, print_interval=1)

print("--- AlphaZero: After Training (vs Random) ---")
print(evaluate_vs_random(az_agent, TicTacToe, 20))

--- AlphaZero: Before Training (vs Random) ---
{'win': 15, 'loss': 2, 'draw': 3}
--- AlphaZero: Training ---
Training AlphaZero for 30 epochs...
Epoch 1/30 | Loss: 2.3548 | Policy: 1.9083 | Value: 0.4465 | Buffer: 200
Epoch 2/30 | Loss: 1.8773 | Policy: 1.6169 | Value: 0.2604 | Buffer: 419
Epoch 3/30 | Loss: 1.8151 | Policy: 1.4651 | Value: 0.3501 | Buffer: 623
Epoch 4/30 | Loss: 1.6794 | Policy: 1.3434 | Value: 0.3361 | Buffer: 834
Epoch 5/30 | Loss: 1.5969 | Policy: 1.2455 | Value: 0.3513 | Buffer: 1048
Epoch 6/30 | Loss: 1.5538 | Policy: 1.2251 | Value: 0.3287 | Buffer: 1294
Epoch 7/30 | Loss: 1.4602 | Policy: 1.1460 | Value: 0.3142 | Buffer: 1528
Epoch 8/30 | Loss: 1.4310 | Policy: 1.1253 | Value: 0.3056 | Buffer: 1782
Epoch 9/30 | Loss: 1.3729 | Policy: 1.0931 | Value: 0.2798 | Buffer: 2036
Epoch 10/30 | Loss: 1.3773 | Policy: 1.0836 | Value: 0.2937 | Buffer: 2272
Epoch 11/30 | Loss: 1.3060 | Policy: 1.0464 | Value: 0.2596 | Buffer: 2513
Epoch 12/30 | Loss: 1.2903 | Policy: 1.0372

In [ ]:
muzero_agent = MuZeroAgent(
    TicTacToe,
    num_simulations=50,
    lr=0.001,
    batch_size=32,
    num_unroll_steps=3,
    td_steps=3,
    discount=1.0,
    latent_dim=64,
    num_blocks=3
)

print("--- MuZero: Before Training (vs Random) ---")
print(evaluate_vs_random(muzero_agent, TicTacToe, 20))

print("--- MuZero: Training ---")
muzero_agent.train(num_epochs=30, games_per_epoch=30, steps_per_epoch=100, print_interval=1)

print("--- MuZero: After Training (vs Random) ---")
print(evaluate_vs_random(muzero_agent, TicTacToe, 20))

--- MuZero: Before Training (vs Random) ---
{'win': 18, 'loss': 0, 'draw': 2}
--- MuZero: Training ---
Training MuZero for 30 epochs...
Epoch 1/30 | Loss: 4.6728 | V: 1.8247 | R: 0.3052 | P: 6.7530 | Buffer: 30
Epoch 2/30 | Loss: 4.3420 | V: 1.3611 | R: 0.2846 | P: 6.5174 | Buffer: 60
Epoch 3/30 | Loss: 4.1781 | V: 1.1812 | R: 0.2652 | P: 6.3832 | Buffer: 90
Epoch 4/30 | Loss: 4.1297 | V: 1.1704 | R: 0.2512 | P: 6.3525 | Buffer: 120
Epoch 5/30 | Loss: 4.0847 | V: 1.2389 | R: 0.2478 | P: 6.2198 | Buffer: 150
Epoch 6/30 | Loss: 4.0830 | V: 1.3415 | R: 0.2378 | P: 6.1541 | Buffer: 180
Epoch 7/30 | Loss: 4.0153 | V: 1.3744 | R: 0.2329 | P: 6.0250 | Buffer: 210
Epoch 8/30 | Loss: 3.9608 | V: 1.4317 | R: 0.2368 | P: 5.8628 | Buffer: 240
Epoch 9/30 | Loss: 3.8740 | V: 1.4322 | R: 0.2314 | P: 5.6942 | Buffer: 270
Epoch 10/30 | Loss: 3.7837 | V: 1.4546 | R: 0.2138 | P: 5.5175 | Buffer: 300
Epoch 11/30 | Loss: 3.6964 | V: 1.4845 | R: 0.2117 | P: 5.3163 | Buffer: 330
Epoch 12/30 | Loss: 3.6096 | 

In [ ]:
play_match(az_agent, muzero_agent, TicTacToe, num_games=20, temperature=0.0, verbose=False)

{'agent1_wins': 0,
 'agent2_wins': 0,
 'draws': 20,
 'agent1_time': 3.769474506378174,
 'agent2_time': 3.6199381351470947,
 'total_moves': 180}